# Chapter 5 — Sparsification: From $k$-NN to Differentiable Top-$k$

Companion notebook to Chapter 5.

Reproduces:
- Figure 5.1: Sparsity patterns of all five sparsifiers on a common dense affinity.
- Figure 5.2: Sparsemax / entmax / softmax weight distributions.
- Figure 5.3: Sparsity-vs-density tradeoff on a kernel-NW prediction task.
- Figure 5.4: Numerical verification of the Santos 2026 equivalence (sparsemax = NW + Epanechnikov).


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from tabkernels.sparsifiers import (
    KNNSparsifier, SparsemaxSparsifier, EntmaxSparsifier,
    SinkhornTopKSparsifier, EpsilonBallSparsifier, sparsemax, entmax_alpha,
)

torch.manual_seed(42); np.random.seed(42)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## Figure 5.1: Sparsity patterns of the five sparsifiers

In [ ]:
# Build a dense affinity from a 2D point set: RBF on Euclidean distances.
N = 30
X = torch.randn(N, 2)
sq = ((X[:, None] - X[None, :]) ** 2).sum(-1)
W = torch.exp(-sq / 1.0)

K = 5
results = {
    'Dense': W.clone(),
    'Hard k-NN (k=5)': KNNSparsifier(k=K)(W),
    'Mutual k-NN': KNNSparsifier(k=K, mutual=True)(W),
    'Sparsemax (rows)': sparsemax(W * 5.0, dim=-1),  # scale up so threshold trims
    'Entmax (alpha=1.5)': entmax_alpha(W * 5.0, alpha=1.5),
    'Eps-ball (eps=0.3)': EpsilonBallSparsifier(epsilon=0.3)(W),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (name, M) in zip(axes.ravel(), results.items()):
    ax.imshow(M.detach(), cmap='viridis', aspect='auto')
    nz = (M > 1e-6).float().mean().item()
    ax.set_title(f'{name}\n(non-zero fraction: {nz:.2f})')
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Figure 5.1: Sparsity patterns of the five sparsifiers')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_05_01_sparsity_patterns.pdf', bbox_inches='tight')
plt.show()

## Figure 5.2: Softmax / Entmax-1.5 / Sparsemax on a single row

In [ ]:
scores = torch.tensor([[3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5]])
out_softmax = torch.softmax(scores, dim=-1).squeeze().numpy()
out_15 = entmax_alpha(scores, alpha=1.5).squeeze().numpy()
out_sparse = sparsemax(scores, dim=-1).squeeze().numpy()

x = np.arange(len(out_softmax))
fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
width = 0.27
ax.bar(x - width, out_softmax, width, label='softmax (α=1)', color='C0')
ax.bar(x, out_15, width, label='entmax (α=1.5)', color='C1')
ax.bar(x + width, out_sparse, width, label='sparsemax (α=2)', color='C2')
ax.set_xticks(x); ax.set_xticklabels([f'{s:.1f}' for s in scores.squeeze().tolist()], rotation=0, fontsize=8)
ax.set_xlabel('score'); ax.set_ylabel('probability')
ax.legend()
ax.set_title('Figure 5.2: Output of softmax, entmax (α=1.5), and sparsemax on a single score vector')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_05_02_sparsemax_entmax_softmax.pdf', bbox_inches='tight')
plt.show()
print(f'softmax non-zero fraction: {(out_softmax > 1e-6).mean():.2f}')
print(f'entmax-1.5 non-zero fraction: {(out_15 > 1e-6).mean():.2f}')
print(f'sparsemax non-zero fraction: {(out_sparse > 1e-6).mean():.2f}')

## Figure 5.3: Sparsity-vs-density tradeoff on kernel-NW prediction

In [ ]:
# Generate 1D regression data; predict via kernel-NW with various sparsifiers.
N_tr, N_te = 80, 50
x_tr = torch.linspace(-3, 3, N_tr).unsqueeze(-1)
y_tr = torch.sin(x_tr.squeeze()) + 0.2 * torch.randn(N_tr)
x_te = torch.linspace(-3, 3, N_te).unsqueeze(-1)
y_te = torch.sin(x_te.squeeze())

def nw_predict(W_te_tr, y_tr):
    denom = W_te_tr.sum(dim=-1, keepdim=True).clamp_min(1e-9)
    return (W_te_tr / denom) @ y_tr

def affinity(x_q, x_t, sigma=0.5):
    sq = ((x_q[:, None] - x_t[None, :]) ** 2).sum(-1)
    return torch.exp(-sq / sigma**2)

W_dense = affinity(x_te, x_tr, sigma=0.5)
ks = [1, 3, 5, 10, 20, 40, 80]
mse_knn = []
for k in ks:
    # Hard k-NN per row
    _, idx = W_dense.topk(min(k, N_tr), dim=-1)
    mask = torch.zeros_like(W_dense)
    mask.scatter_(1, idx, 1.0)
    W_k = W_dense * mask
    pred = nw_predict(W_k, y_tr)
    mse_knn.append(((pred - y_te) ** 2).mean().item())

alphas = [1.0, 1.25, 1.5, 1.75, 2.0]
mse_entmax = []
for a in alphas:
    weights = entmax_alpha(W_dense * 5.0, alpha=a) if a > 1.0 else torch.softmax(W_dense * 5.0, dim=-1)
    pred = weights @ y_tr
    mse_entmax.append(((pred - y_te) ** 2).mean().item())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(ks, mse_knn, 'o-'); axes[0].set_xlabel('k (hard k-NN)'); axes[0].set_ylabel('test MSE')
axes[0].set_title('Hard k-NN sparsity'); axes[0].grid(alpha=0.3)
axes[1].plot(alphas, mse_entmax, 'o-'); axes[1].set_xlabel('entmax alpha'); axes[1].set_ylabel('test MSE')
axes[1].set_title('Entmax sparsity'); axes[1].grid(alpha=0.3)
plt.suptitle('Figure 5.3: Sparsity-vs-density tradeoff on kernel-NW prediction')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_05_03_tradeoff.pdf', bbox_inches='tight')
plt.show()

## Figure 5.4: Santos 2026 equivalence (sparsemax = NW + Epanechnikov)

From Eq. (5.7), `sparsemax(-||x_q - x_i||^2 / 2h^2)` should match Nadaraya–Watson with the Epanechnikov kernel `max(0, 1 - r^2/h^2)`.

In [ ]:
torch.manual_seed(0)
X = torch.randn(50, 4)
h = 1.5
sq_dist = ((X[:, None] - X[None, :]) ** 2).sum(-1)

# Sparsemax of negative scaled distances.
scores = -sq_dist / (2 * h ** 2)
weights_sparsemax = sparsemax(scores, dim=-1)

# Epanechnikov NW.
epi = (1 - sq_dist / h ** 2).clamp_min(0.0)
weights_epanechnikov = epi / epi.sum(dim=-1, keepdim=True).clamp_min(1e-12)

# Plot 4 rows side-by-side.
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, idx in zip(axes.ravel(), [0, 5, 10, 25]):
    ax.plot(weights_sparsemax[idx].detach(), 'o-', label='sparsemax', alpha=0.7)
    ax.plot(weights_epanechnikov[idx].detach(), 'x--', label='Epanechnikov NW', alpha=0.7)
    ax.set_title(f'Row {idx}'); ax.set_xlabel('column'); ax.set_ylabel('weight')
    ax.legend(fontsize=8)
plt.suptitle('Figure 5.4: Santos 2026 equivalence — sparsemax of squared-distance scores vs. NW with Epanechnikov')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_05_04_santos_equivalence.pdf', bbox_inches='tight')
plt.show()

# Numerical similarity at the support level.
sparsemax_zeros = (weights_sparsemax == 0)
epi_zeros = (weights_epanechnikov == 0)
overlap = (sparsemax_zeros == epi_zeros).float().mean().item()
print(f'Support overlap of sparsemax and Epanechnikov NW: {overlap*100:.1f}%')

**End of notebook.** Reproduces 4 figures: sparsity patterns, sparsemax/entmax/softmax distributions, sparsity-vs-density tradeoff, and the Santos 2026 equivalence.